In [1]:
import cv2
import numpy as np

# Paths to models
FACE_PROTO = "deploy.prototxt"
FACE_MODEL = "res10_300x300_ssd_iter_140000.caffemodel"
AGE_PROTO = "age_deploy.prototxt"
AGE_MODEL = "age_net.caffemodel"

# Age range buckets
AGE_BUCKETS = ["(0-2)", "(4-6)", "(8-12)", "(15-20)", "(25-32)", "(38-43)", "(48-53)", "(60-100)"]

# Load models
face_net = cv2.dnn.readNet(FACE_MODEL, FACE_PROTO)
age_net = cv2.dnn.readNet(AGE_MODEL, AGE_PROTO)

# Function to detect faces
def detect_faces(net, frame, conf_threshold=0.7):
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104, 117, 123], True, False)
    net.setInput(blob)
    detections = net.forward()
    h, w = frame.shape[:2]
    boxes = []

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > conf_threshold:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            boxes.append(box.astype("int"))

    return boxes

# Function to predict age
def predict_age(age_net, face):
    blob = cv2.dnn.blobFromImage(face, 1.0, (227, 227), [78.4263377603, 87.7689143744, 114.895847746], swapRB=False)
    age_net.setInput(blob)
    age_predictions = age_net.forward()
    age = AGE_BUCKETS[age_predictions[0].argmax()]
    return age

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_resized = cv2.resize(frame, (640, 480))
    face_boxes = detect_faces(face_net, frame_resized)

    for box in face_boxes:
        x1, y1, x2, y2 = box
        face = frame_resized[y1:y2, x1:x2]

        try:
            face_blob = cv2.resize(face, (227, 227))
            age = predict_age(age_net, face_blob)
            label = f"Age: {age}"
            cv2.rectangle(frame_resized, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame_resized, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        except Exception as e:
            print(f"Error processing face: {e}")

    cv2.imshow("Age Detector", frame_resized)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\dnn\src\caffe\caffe_io.cpp:1126: error: (-2:Unspecified error) FAILED: fs.is_open(). Can't open "deploy.prototxt" in function 'cv::dnn::ReadProtoFromTextFile'


In [2]:
!pip install opencv-python opencv-python-headless numpy tensorflow


   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.1/38.8 MB 656.4 kB/s eta 0:00:59
   ---------------------------------------- 0.2/38.8 MB 1.3 MB/s eta 0:00:30
   ---------------------------------------- 0.5/38.8 MB 2.7 MB/s eta 0:00:15
   ---------------------------------------- 0.5/38.8 MB 2.7 MB/s eta 0:00:15
    --------------------------------------- 1.0/38.8 MB 3.6 MB/s eta 0:00:11
   - -------------------------------------- 1.6/38.8 MB 5.0 MB/s eta 0:00:08
   -- ------------------------------------- 2.4/38.8 MB 7.1 MB/s eta 0:00:06
   --- ------------------------------------ 3.1/38.8 MB 7.7 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/38.8 MB 9.3 MB/s eta 0:00:04
   ----- ---------------------------------- 4.9/38.8 MB 9.8 MB/s eta 0:00:04
   ------ --------

ERROR: Exception:
Traceback (most recent call last):
  File "E:\Anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "E:\Anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "E:\Anaconda\Lib\socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\Anaconda\Lib\ssl.py", line 1252, in recv_int

In [ ]:
pip install opencv-python

In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Load pre-trained emotion detection model
model_path = "emotion_model.h5"  # Path to the model file
emotion_model = load_model(model_path)

# Define emotion labels (based on FER-2013 dataset)
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Function to detect emotions
def detect_emotions(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert frame to grayscale
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        # Extract Region of Interest (ROI)
        roi_gray = gray[y:y+h, x:x+w]
        roi_gray = cv2.resize(roi_gray, (48, 48))
        roi_gray = roi_gray.astype('float') / 255.0  # Normalize the ROI
        roi_gray = img_to_array(roi_gray)
        roi_gray = np.expand_dims(roi_gray, axis=0)

        # Predict emotion
        preds = emotion_model.predict(roi_gray)[0]
        emotion = emotion_labels[np.argmax(preds)]
        confidence = np.max(preds)

        # Draw rectangle around face
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        # Display emotion label and confidence
        text = f"{emotion}: {confidence*100:.2f}%"
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return frame

# Real-time webcam emotion detection
def run_emotion_detector():
    cap = cv2.VideoCapture(0)  # Open default webcam
    print("Starting real-time emotion detection. Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect emotions in the current frame
        frame_with_emotions = detect_emotions(frame)

        # Display the frame
        cv2.imshow('Emotion Detector', frame_with_emotions)

        # Press 'q' to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run the detector
if __name__ == "__main__":
    run_emotion_detector()


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'emotion_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [2]:
import cv2
import numpy as np

# Load the pre-trained age detection model and face detection cascade
age_net = cv2.dnn.readNetFromCaffe(
    "deploy_age.prototxt", "age_net.caffemodel"  # Model files
)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Age groups as per the pre-trained model
age_groups = ['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']

# Function to detect age
def detect_age(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        # Extract face ROI
        face_roi = frame[y:y+h, x:x+w]
        blob = cv2.dnn.blobFromImage(face_roi, 1.0, (227, 227), (78.4263377603, 87.7689143744, 114.895847746), swapRB=False)
        
        # Predict age
        age_net.setInput(blob)
        age_preds = age_net.forward()
        age = age_groups[np.argmax(age_preds)]

        # Draw rectangle and display age
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, f'Age: {age}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return frame

# Run real-time age detection
def run_age_detector():
    cap = cv2.VideoCapture(0)  # Open default webcam
    print("Starting Age Detection. Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect age in the current frame
        frame_with_age = detect_age(frame)

        # Display the frame
        cv2.imshow('Age Detector', frame_with_age)

        # Exit on pressing 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_age_detector()


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\dnn\src\caffe\caffe_io.cpp:1126: error: (-2:Unspecified error) FAILED: fs.is_open(). Can't open "deploy_age.prototxt" in function 'cv::dnn::ReadProtoFromTextFile'


In [7]:
import cv2

# Load Haar cascades for face and eye detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# Function to detect faces and eyes
def detect_faces_and_eyes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert frame to grayscale

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        # Draw rectangle around the face
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        # Region of interest for eyes within the detected face
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        # Detect eyes within the face
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            # Draw rectangle around the eyes
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

    return frame

# Real-time face and eye detection using webcam
def run_detection():
    cap = cv2.VideoCapture(0)  # Open the webcam
    print("Starting Face and Eye Detection. Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect faces and eyes in the frame
        processed_frame = detect_faces_and_eyes(frame)

        # Display the frame
        cv2.imshow('Face and Eye Detection', processed_frame)

        # Exit when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_detection()


Starting Face and Eye Detection. Press 'q' to quit.


In [5]:
import cv2

# Load Haar cascades for face and eye detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# Function to detect faces and eyes
def detect_faces_and_eyes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # Convert frame to grayscale

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        # Draw rectangle around the face
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        # Region of interest for eyes within the detected face
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        # Detect eyes within the face
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            # Draw rectangle around the eyes
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

    return frame

# Real-time face and eye detection using webcam
def run_detection():
    cap = cv2.VideoCapture(0)  # Open the webcam
    print("Starting Face and Eye Detection. Press 'q' to quit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect faces and eyes in the frame
        processed_frame = detect_faces_and_eyes(frame)

        # Display the frame
        cv2.imshow('Face and Eye Detection', processed_frame)

        # Exit when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_detection()


Starting Face and Eye Detection. Press 'q' to quit.


In [8]:
a=22
print(a)

22


In [9]:
a=44
a

44

In [10]:
type(a)

int

In [11]:
b=[1,2,3,4,5]
b

[1, 2, 3, 4, 5]

In [12]:
type(b)

list

In [13]:
c={1:"apple",2:"mango",3:"bannaa"}
c

{1: 'apple', 2: 'mango', 3: 'bannaa'}

In [14]:
c.items()

dict_items([(1, 'apple'), (2, 'mango'), (3, 'bannaa')])

In [15]:
c.values()

dict_values(['apple', 'mango', 'bannaa'])

In [16]:
c.get(1)

'apple'

In [17]:
c.get(2)

'mango'

In [18]:
c.get(3)

'bannaa'

In [19]:
c.get("bannaa")

In [20]:
c

{1: 'apple', 2: 'mango', 3: 'bannaa'}

In [21]:
c.get(2)

'mango'

In [22]:
c.get(1)

'apple'

In [23]:
c.get(3)

'bannaa'

In [24]:
c.items()

dict_items([(1, 'apple'), (2, 'mango'), (3, 'bannaa')])

In [25]:
c.keys()


dict_keys([1, 2, 3])

In [26]:
c.keys()

dict_keys([1, 2, 3])

In [27]:
c.values()

dict_values(['apple', 'mango', 'bannaa'])

In [28]:
c

{1: 'apple', 2: 'mango', 3: 'bannaa'}

In [29]:
c

{1: 'apple', 2: 'mango', 3: 'bannaa'}

In [30]:
print(type(c))

<class 'dict'>


In [31]:
print(c)

{1: 'apple', 2: 'mango', 3: 'bannaa'}


In [32]:
print(type(c))

<class 'dict'>


In [33]:
d=[1,2,3,4,"apple",44]
d

[1, 2, 3, 4, 'apple', 44]

In [34]:
type(d)

list

In [35]:
d

[1, 2, 3, 4, 'apple', 44]

In [36]:
print(type(d))

<class 'list'>


In [37]:
d

[1, 2, 3, 4, 'apple', 44]

In [38]:
type(d)

list

In [39]:
e=(1,2,"apple","mango")
e

(1, 2, 'apple', 'mango')

In [40]:
type(e)

tuple

In [ ]:
e.